# SynBioS MoE：端到端 smoke test

这个 notebook 按真实 CLI 入口依次覆盖：数据准备 → 主训练 → 主模型评估 → 单 probe 后训练与独立 validation → P/Q pipeline → router 分析与结果汇总。每个阶段都是独立单元格，方便逐段运行和定位问题。

> 默认是结构性 smoke test：100 人、主训练 2 steps、每种 probe 3 steps。它只能验证调用链路，不能作为论文实验结果。正式实验仍使用 `configs/synbios_moe/runs/*.yaml`。

## 阶段 0：定位项目并设置测试参数

In [ ]:
import json
import os
from pathlib import Path

import torch

ROOT = Path.cwd().resolve()
if not (ROOT / 'scripts' / 'synbios_moe.py').is_file():
    if (ROOT.parent / 'scripts' / 'synbios_moe.py').is_file():
        ROOT = ROOT.parent
    else:
        raise RuntimeError('请从 mini-train-sys/ 或 mini-train-sys/tests/ 启动 notebook')
os.chdir(ROOT)

DATA_DIR = Path('artifacts/synbios_moe/smoke')
CHECKPOINT_ROOT = Path('artifacts/synbios_moe/checkpoints/synbios_moe_smoke')
PROBE_CACHE = DATA_DIR / 'probe_cache'
RESULT_DIR = Path('artifacts/synbios_moe/results/notebook_smoke')
EXP_LOG_DIR = RESULT_DIR / 'operation_logs'
PIPELINE_DIR = RESULT_DIR / 'probe_pipeline'
PIPELINE_CONFIG = 'configs/synbios_moe/probe_pipeline_notebook_smoke.yaml'
MODEL_CONFIG = 'configs/synbios_moe_smoke_model.yaml'
TRAIN_CONFIG = 'configs/synbios_moe_smoke_pretrain.yaml'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ATTRIBUTE = 'birth_city'
NUM_PEOPLE = 100
TRAIN_STEPS = 2
PROBE_STEPS = 3
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print('repo       :', ROOT)
print('device     :', DEVICE)
print('data       :', DATA_DIR)
print('results    :', RESULT_DIR)

In [ ]:
# Windows 环境检查也直接通过 notebook 的 ! 命令执行。
!python --version
!python -c "import torch, tiktoken; print('torch=', torch.__version__, 'cuda=', torch.cuda.is_available())"

## 阶段 1：生成 biography 并预处理成 token shards

`prepare` 会生成 profiles、biographies、manifest 和训练直接读取的 token shards。

In [ ]:
!python scripts/synbios_moe.py prepare --output {DATA_DIR.as_posix()} --variant single --num-people {NUM_PEOPLE} --seed 1337 --max-shard-tokens 100000 --log-dir {EXP_LOG_DIR.as_posix()} --log-interval 1 --tensorboard
!python scripts/synbios_moe.py cache-probes --data {DATA_DIR.as_posix()} --output {PROBE_CACHE.as_posix()} --force

In [ ]:
required_data_files = [
    DATA_DIR / 'profiles.jsonl',
    DATA_DIR / 'biographies.jsonl',
    DATA_DIR / 'manifest.json',
    DATA_DIR / 'token_shards/manifest.json',
    PROBE_CACHE / 'manifest.json',
]
assert all(path.is_file() for path in required_data_files), required_data_files
dataset_manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
token_manifest = json.loads((DATA_DIR / 'token_shards/manifest.json').read_text(encoding='utf-8'))
print('dataset:', dataset_manifest)
print('token shards:', token_manifest)

## 阶段 2：主训练（pretraining）

这里调用正式训练入口，但用 `--smoke-steps` 缩短为 2 steps。checkpoint 仍保存模型与 Adam 等恢复状态；后续只读取导出的 `model.pt`。

In [ ]:
!python scripts/train.py --device {DEVICE} --smoke-steps {TRAIN_STEPS} --config {TRAIN_CONFIG} --model-config {MODEL_CONFIG}

In [ ]:
checkpoint_candidates = [
    path for path in CHECKPOINT_ROOT.glob('epoch_*_step_*')
    if (path / 'COMMITTED').is_file() and (path / 'model.pt').is_file()
]
assert checkpoint_candidates, f'没有找到可用 checkpoint: {CHECKPOINT_ROOT}'
CHECKPOINT = max(checkpoint_candidates, key=lambda path: path.stat().st_mtime)
print('checkpoint:', CHECKPOINT)
print('files:', sorted(path.name for path in CHECKPOINT.iterdir()))
assert (CHECKPOINT / 'runtime.pt').is_file(), '缺少 trainer/Adam 恢复元数据'
assert (CHECKPOINT / 'distributed').is_dir(), '缺少 DCP 模型与 Adam 状态'
assert (CHECKPOINT / 'model.pt').is_file(), '缺少供单卡推理/probe 使用的权重导出'

## 阶段 3：生成主模型评估结果

先做 teacher-forced attribute-token accuracy，输出独立 JSON。

In [ ]:
EVAL_JSON = RESULT_DIR / 'pretrain_attribute_accuracy.json'
!python scripts/synbios_moe.py evaluate --data {DATA_DIR.as_posix()} --model-config {MODEL_CONFIG} --checkpoint {CHECKPOINT.as_posix()} --device {DEVICE} --examples 20 --batch-size 4 --output {EVAL_JSON.as_posix()} --log-dir {EXP_LOG_DIR.as_posix()} --log-interval 1 --tensorboard

## 阶段 4A：单个 P-probe 后训练与指标监控

冻结主模型，只训练低秩 P-probe。`--log-interval 1` 使 smoke test 每一步都记录 accuracy、逐位置 accuracy、grad norm、data wait 和 step time。

In [ ]:
P_JSON = RESULT_DIR / f'p_{ATTRIBUTE}_first.json'
!python scripts/synbios_moe.py probe --kind p --data {DATA_DIR.as_posix()} --probe-cache {PROBE_CACHE.as_posix()} --model-config {MODEL_CONFIG} --checkpoint {CHECKPOINT.as_posix()} --attribute {ATTRIBUTE} --target first --device {DEVICE} --steps {PROBE_STEPS} --batch-size 8 --output {P_JSON.as_posix()} --log-dir {EXP_LOG_DIR.as_posix()} --log-interval 1 --tensorboard

## 阶段 4B：独立恢复 P-probe 并运行 validation

这一步重新加载刚保存的 probe 权重和主模型，只做 held-out validation；它不需要主训练 Adam，也不需要 probe optimizer。

In [ ]:
P_VAL_JSON = RESULT_DIR / f'p_{ATTRIBUTE}_first_validation.json'
!python scripts/synbios_moe.py validate-probe --data {DATA_DIR.as_posix()} --probe-cache {PROBE_CACHE.as_posix()} --model-config {MODEL_CONFIG} --checkpoint {CHECKPOINT.as_posix()} --probe-checkpoint {P_JSON.with_suffix('.pt').as_posix()} --device {DEVICE} --batch-size 8 --output {P_VAL_JSON.as_posix()} --log-dir {EXP_LOG_DIR.as_posix()} --log-interval 1 --tensorboard

## 阶段 4C：运行 P/Q probe pipeline

notebook 专用配置只运行 3 steps 的 P/Q 两个任务。它仍走正式调度器，覆盖任务队列、独立 validation、summary、JSONL/TensorBoard 和心跳；`--devices` 可替换成任意显式 GPU 列表。

In [ ]:
!python scripts/synbios_moe.py probe-pipeline --stage smoke --pipeline-config {PIPELINE_CONFIG} --data {DATA_DIR.as_posix()} --probe-cache {PROBE_CACHE.as_posix()} --model-config {MODEL_CONFIG} --checkpoint {CHECKPOINT.as_posix()} --output {PIPELINE_DIR.as_posix()} --devices {DEVICE} --skip-gate --log-interval 1 --heartbeat-seconds 0.1 --tensorboard --quiet-workers

## 阶段 4D：检查 pipeline 状态和监控事件

先检查顶层状态，再检查 started/heartbeat/finished 事件。正式长任务默认使用 30 秒心跳。

In [ ]:
PIPELINE_STAGE = PIPELINE_DIR / 'smoke'
pipeline_state = json.loads((PIPELINE_STAGE / 'pipeline.json').read_text(encoding='utf-8'))
pipeline_events = [json.loads(line) for line in (PIPELINE_STAGE / 'pipeline_events.jsonl').read_text(encoding='utf-8').splitlines()]
assert pipeline_state['status'] == 'completed', pipeline_state
identity = pipeline_state['identity']
assert identity['protocol_version'] == 2
assert {'data_manifest_sha256', 'probe_cache_manifest_sha256', 'model_config_sha256', 'checkpoint_model_sha256'} <= identity.keys()
actions = {event['action'] for event in pipeline_events}
assert actions >= {'started', 'finished'}
assert len(pipeline_state['training']) == 2 and len(pipeline_state['validation']) == 2
assert all(item['status'] in {'completed', 'skipped_existing'} for item in pipeline_state['training'] + pipeline_state['validation'])
if any(item['status'] == 'completed' for item in pipeline_state['training'] + pipeline_state['validation']):
    assert 'heartbeat' in actions, '新启动的子进程应产生 heartbeat'
print('pipeline monitoring:', pipeline_state['monitoring_log_dir'])
print('pipeline events:', len(pipeline_events))

## 阶段 5：router 分析

不再训练参数，只生成 expert load、entropy 和属性标签 NMI。

In [ ]:
ROUTER_JSON = RESULT_DIR / f'router_{ATTRIBUTE}.json'
!python scripts/synbios_moe.py analyze --data {DATA_DIR.as_posix()} --probe-cache {PROBE_CACHE.as_posix()} --model-config {MODEL_CONFIG} --checkpoint {CHECKPOINT.as_posix()} --attribute {ATTRIBUTE} --target first --device {DEVICE} --examples 20 --output {ROUTER_JSON.as_posix()} --log-dir {EXP_LOG_DIR.as_posix()} --log-interval 1 --tensorboard

## 阶段 6：校验并汇总结果数据

In [ ]:
result_files = {
    'pretrain_evaluation': EVAL_JSON,
    'p_probe': P_JSON,
    'p_probe_validation': P_VAL_JSON,
    'router_analysis': ROUTER_JSON,
}
assert all(path.is_file() for path in result_files.values()), result_files
assert P_JSON.with_suffix('.pt').is_file()
assert (PIPELINE_STAGE / 'summary/summary.json').is_file()
assert (PIPELINE_STAGE / 'summary/summary.csv').is_file()
event_logs = list(EXP_LOG_DIR.rglob('events.jsonl'))
tensorboard_events = list(EXP_LOG_DIR.rglob('events.out.tfevents.*'))
assert len(event_logs) >= 5, event_logs
assert len(tensorboard_events) >= 5, tensorboard_events
summary = {name: json.loads(path.read_text(encoding='utf-8')) for name, path in result_files.items()}
assert all(payload.get('monitoring') for payload in summary.values()), '每阶段都应保存监控摘要'
probe_payload = summary['p_probe']
required_probe_metrics = {'accuracy', 'accuracy_by_position', 'grad_norm', 'data_wait_ms', 'data_wait_percent', 'step_time_ms'}
assert required_probe_metrics <= probe_payload['loss_curve'][-1].keys(), probe_payload['loss_curve'][-1]
structured_events = [json.loads(line) for path in event_logs for line in path.read_text(encoding='utf-8').splitlines()]
probe_train_events = [event for event in structured_events if event.get('event') == 'probe_train']
probe_validation_events = [event for event in structured_events if event.get('event') == 'probe_validation']
assert any(required_probe_metrics <= event.keys() for event in probe_train_events), probe_train_events[-3:]
assert any('accuracy_by_position_running' in event for event in probe_validation_events), probe_validation_events[-3:]
summary_path = RESULT_DIR / 'notebook_summary.json'
summary_path.write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
print('PASS: SynBioS MoE 端到端调用链完整')
print('summary:', summary_path)
print('result files:', [str(path) for path in sorted(RESULT_DIR.iterdir())])

## 正式实验：仍然按阶段运行（不要在 smoke test 中直接执行）

下面把正式实验继续拆成数据、主训练、probe cache、smoke、pilot、formal 和比较单元格。先设置一组 `FORMAL_*` 路径，再逐格取消命令前的 `#`；改变 `--num-gpus` 即可使用任意卡数。

In [ ]:
# 正式阶段 1A：single 数据
# !python scripts/synbios_moe.py prepare --output artifacts/synbios_moe/single --variant single --num-people 100000 --seed 1337

In [ ]:
# 正式阶段 1B：multi5+permute 数据（相同 seed 保证人物事实表一致）
# !python scripts/synbios_moe.py prepare --output artifacts/synbios_moe/multi5_permute --variant multi5+permute --num-people 100000 --seed 1337

In [ ]:
# 正式阶段 2A：single 主训练；中断恢复时在末尾追加 --resume latest
# !python scripts/train.py --device cuda --config configs/synbios_moe/runs/single_single.yaml --model-config configs/synbios_moe/model.yaml

In [ ]:
# 正式阶段 2B：multi5+permute 主训练；中断恢复时追加 --resume latest
# !python scripts/train.py --device cuda --config configs/synbios_moe/runs/multi5_permute_single.yaml --model-config configs/synbios_moe/model.yaml

### 正式阶段 3：选择一个 run 并生成 probe cache

先填写该 run 的最终主模型 checkpoint。完成 single 全部 probe 阶段后，再把这组变量切换到 multi5_permute 重跑。

In [ ]:
# FORMAL_DATA = Path('artifacts/synbios_moe/single')
# FORMAL_CACHE = FORMAL_DATA / 'probe_cache'
# FORMAL_CHECKPOINT = Path('artifacts/synbios_moe/checkpoints/single_single/<committed-checkpoint>')
# FORMAL_PIPELINE = Path('artifacts/synbios_moe/results/single_single/probe_pipeline')
# !python scripts/synbios_moe.py cache-probes --data {FORMAL_DATA.as_posix()} --output {FORMAL_CACHE.as_posix()} --require-coverage

### 正式阶段 4：probe smoke

先用两个任务检查 checkpoint、数据、显存和监控产物。

In [ ]:
# !python scripts/synbios_moe.py probe-pipeline --stage smoke --data {FORMAL_DATA.as_posix()} --probe-cache {FORMAL_CACHE.as_posix()} --model-config configs/synbios_moe/model.yaml --checkpoint {FORMAL_CHECKPOINT.as_posix()} --output {FORMAL_PIPELINE.as_posix()} --devices auto --num-gpus 1 --require-coverage --log-interval 10 --heartbeat-seconds 30

### 正式阶段 5：probe pilot

pilot 覆盖全部 22 个任务，用于估时和检查 loss/accuracy/grad norm 是否正常。

In [ ]:
# !python scripts/synbios_moe.py probe-pipeline --stage pilot --data {FORMAL_DATA.as_posix()} --probe-cache {FORMAL_CACHE.as_posix()} --model-config configs/synbios_moe/model.yaml --checkpoint {FORMAL_CHECKPOINT.as_posix()} --output {FORMAL_PIPELINE.as_posix()} --devices auto --num-gpus 1 --require-coverage --log-interval 100 --heartbeat-seconds 30

### 正式阶段 6：probe formal

formal 只有在 pilot 的 `pipeline.json` 为 completed 后才启动。服务器有更多 GPU 时只修改 `--num-gpus`。

In [ ]:
# !python scripts/synbios_moe.py probe-pipeline --stage formal --data {FORMAL_DATA.as_posix()} --probe-cache {FORMAL_CACHE.as_posix()} --model-config configs/synbios_moe/model.yaml --checkpoint {FORMAL_CHECKPOINT.as_posix()} --output {FORMAL_PIPELINE.as_posix()} --devices auto --num-gpus 1 --require-coverage --log-interval 300 --heartbeat-seconds 30

### 正式阶段 7：single 与 multi5+permute 后处理比较

In [ ]:
# !python scripts/synbios_moe.py summarize-probes --run single=artifacts/synbios_moe/results/single_single/probe_pipeline/formal/validation --run multi5_permute=artifacts/synbios_moe/results/multi5_permute_single/probe_pipeline/formal/validation --output artifacts/synbios_moe/results/comparison

## Smoke resume check

Run this after the smoke stages above. It restores model, Adam, LR, counters and RNG from the committed checkpoint, then performs one additional optimizer step.

In [ ]:
!python scripts/train.py --device {DEVICE} --smoke-steps 3 --config {TRAIN_CONFIG} --model-config {MODEL_CONFIG} --resume {CHECKPOINT.as_posix()}
resumed = [p for p in CHECKPOINT_ROOT.glob('epoch_*_step_000000003') if (p / 'COMMITTED').is_file()]
assert resumed, 'resume did not produce the expected step-3 checkpoint'
print('PASS: full-state resume checkpoint:', max(resumed, key=lambda p: p.stat().st_mtime))